# Fetching data from SQL database

In [29]:
import mysql.connector
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

conn = mysql.connector.connect(
    host='127.0.0.1',
    port=13306,
    user='root',
    password='secret',
    database='odb'
)

cursor = conn.cursor()

# Function for fetching data from database
def fetch_data_from_db():
    query = "SELECT * FROM terrorism"
    cursor.execute(query)
    result = cursor.fetchall()
    columns = [desc[0] for desc in cursor.description]
    df = pd.DataFrame(result, columns=columns)
    return df


df = fetch_data_from_db()


print(df.head())
print(df.dtypes)

        eventid  year  month  day  country  country_txt  region  \
0  202001010001  2020      1    1      141        Nepal       6   
1  202001010002  2020      1    1      141        Nepal       6   
2  202001010003  2020      1    1      200        Syria      10   
3  202001010005  2020      1    1        4  Afghanistan       6   
4  202001010006  2020      1    1       19   Bangladesh       6   

                   region_txt      city  success  ...  victim_nat_txt  \
0                  South Asia  Jurethum        1  ...           Nepal   
1                  South Asia    Pipira        0  ...           Nepal   
2  Middle East & North Africa     Suluk        1  ...           Syria   
3                  South Asia     Farah        1  ...     Afghanistan   
4                  South Asia     Dhaka        1  ...      Bangladesh   

                                      attacker_group  \
0  Communist Party of Nepal - Maoist (CPN-Maoist-...   
1  Communist Party of Nepal - Maoist (CPN-Maoi

In [ ]:
country_counts = df['country_txt'].value_counts().sort_index().reset_index()
country_counts.columns = ['country', 'count']

# Create scatter plot
#fig = go.Figure(go.Scattergeo())

#fig.update_geos(projection_type='orthographic')
#fig.update_layout(height=400)


import pycountry

def country_to_iso(name):
    try:
        return pycountry.countries.lookup(name).alpha_3  # or use .alpha_2 for 2-letter codes
    except LookupError:
        return 2
    
country_counts['iso_code'] = country_counts['country'].apply(country_to_iso)


print(country_counts)

fig = px.choropleth(country_counts, locations='iso_code', color='count', hover_data=['country', 'count'])

fig.update_geos(projection_type='orthographic')
# Show the figure
fig.show()


                      country  count iso_code
0                 Afghanistan   2604      AFG
1                     Albania      1      ALB
2                     Algeria      4      DZA
3                      Angola      1      AGO
4                   Argentina      5      ARG
..                        ...    ...      ...
96              United States    103      USA
97                  Venezuela      7      VEN
98   West Bank and Gaza Strip     61        2
99                      Yemen    474      YEM
100                  Zimbabwe      1      ZWE

[101 rows x 3 columns]


In [ ]:
# Get list of countries already in the DataFrame
present_countries = country_counts['country'].tolist()

# Prepare rows for missing countries
missing_rows = []

for c in pycountry.countries:
    if c.name not in present_countries:
        missing_rows.append({
            'country': c.name,
            'count': 0,
            'iso_code': c.alpha_3
        })

# Append missing countries
if missing_rows:
    country_counts = pd.concat([country_counts, pd.DataFrame(missing_rows)], ignore_index=True)

# Optional: sort alphabetically or by iso_code
country_counts = country_counts.sort_values(by='country').reset_index(drop=True)

print(country_counts)


fig = px.choropleth(country_counts, locations='iso_code', color='count', hover_data=['country', 'count'], color_continuous_scale=["#fff5f0", "#fcbba1", "#fc9272", "#ef3b2c", "#99000d"])

fig.update_geos(projection_type='orthographic')
# Show the figure
fig.show()


            country  count iso_code
0       Afghanistan   2604      AFG
1           Albania      1      ALB
2           Algeria      4      DZA
3    American Samoa      0      ASM
4           Andorra      0      AND
..              ...    ...      ...
256  Western Sahara      0      ESH
257           Yemen    474      YEM
258          Zambia      0      ZMB
259        Zimbabwe      1      ZWE
260   Åland Islands      0      ALA

[261 rows x 3 columns]
